# **Final Model**

## **Exploratory Data Analysis**

As part of EDA, the following steps were performed:
1. The columns having extremely high variance such as the MachineID column were eliminated because the values in these columns corresponded to the individual samples and had no correspondence with the class labels.
2. Certain columns were observed to have very low variance such as the ProductName columns. In such cases, the distribution of class label for the value of this column being each of the unique values, was checked and if it was obeserved that the presence of this particular unique value has no significant effect on the class label distribution then this column was removed. For example, for the ProductName column, only 2 unique values namely mse(229 instancces) and windows8defender(99771 instances were present), upon further analysis it was found that among the samples with ProductName as mse, the target label was 0 for 54.59% of the samples while it was 1 for the rest 45.41% of samples and among the samples with ProductName as windows8defender, the target label was 0 for 50.53% of cases while it was 1 for the rest 49.46%. Thus, no significant variation in the distribution of target label resulted from the variation in ProductName. Thus, the column ProductName was dropped. However, when this same analysis was done on the DeviceFamily column which is another column with very low variation in values,it was found that the samples with value of this column as Windows.Desktop(99971 instances) had a distribution of 50.53% samples with label 1 and rest 49.46% samples with label 0 while those samples with value of ths column as Windows.Server had a distribution of 72.41% samples with label 0 while 27.59% of samples with label 1. Thus, this column could not be dropped as it significantly impacted the target.
3.  Certain columns such as EngineVersion, AppVersion, SignatureVersion were found to contain several individual pieces of numeric information concatenated together into a categorical data. These individual pieces of information were seperated into distinct columns and then it was observed that the same pieces of information as extracted from these columns were present in certain other columns as numeric data. To investigate this association further, the Analysis of Variance(ANOVA) test was performed and the columns for which the p-value came to be below 0.05, were dropped. Furthermore, in order to obtain the association between categorical columns, Mutual Information was obtained between them and for the pairs of columns with high mutual information values, one of the columns of the pair was dropped.

## **Data Preparation**

The following steps were performed for preparing the data:
1. At first, the null values were imputed across all columns using the mode value of each column. The SimpleImputer from sklearn.imput package was used for this purpose.
2. The columns found unuseful through EDA were dropped.
3. Categorical columns whose values were concatenations of multiple numeric values were splitted into multiple columns, each corresponding to an individual numeric value. Next, the association between these newly obtained columns and the existing columns were analysed through EDA and appropriate columns were dropped.
4. The categorical columns in which, the actual useful information was an embedded numeric one were identified and the categorical values were replaced with the corresponding embedded numeric values. For example, the column PlatformType contained values such as windows10 and windows8. Obviously, the embedded 10 and 8 were the usedful piece of information in these cases and thus they were extracted using the appropriate regular expression
5. The columns containing the dates i.e. DateAS and DateOS were transformed as well to reflect the number of minutes from the earliest date and time present in them. This way, they were converted into a more useful form than their original categorical DD-MM-YYYY form

Reader may note that the majority of data preparation steps were aimed at transforming the categorical features into numerical features as much as possible. The motivation behind this was the fact that irrespective of the model being used(except for CatBoost model) numeric features are always more useful in classification tasks and require no encoding. Moreover, the categorical columns with high number of unique values were successfully eliminated in this way and the remaining categorical columns all contained low number of unique values with the highest being 28(column ChassisType) which made them suitable for OneHotEncoder.

In [305]:
import pandas as pd
import numpy as np
from scipy.stats import f_oneway
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OrdinalEncoder
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

In [306]:
df=pd.read_csv('/kaggle/input/System-Threat-Forecaster/train.csv')
df['MachineID'].nunique()

99835

In [307]:
df['MachineID'].isna().sum()

0

In [308]:
counts = df['MachineID'].value_counts()
dupl = counts[counts>1]
print(dupl)

MachineID
8e0523438a5ca48323f50b2c47f6d31a    2
0ca2c7ebe3921b5e74d8a030646ee9c2    2
43628a7db99daf5bf8ca8a3f36058ea4    2
bb11a6869fb4ec76797e97ae3006c833    2
a5f52a788ea2e509572369585ddd05c9    2
                                   ..
fbd5768ab6271f0e2244ed73add95de2    2
93213d4f191ad2c75b5cbf6b9a51ad28    2
d73897d31665fa1a1ce9148a62569067    2
ae8d51d3eec9bc111233562a572599b0    2
dba13dce0ce152c86fc5a99403b4d73b    2
Name: count, Length: 165, dtype: int64


In [309]:
imputer = SimpleImputer(strategy='most_frequent')  # Mode imputation
df_imputed = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)

df = df_imputed.astype(df.dtypes.to_dict())


In [310]:
print(((df['IsBetaUser']!=1) & (df['IsBetaUser']!=0)).sum())

0


In [311]:
df.dtypes

MachineID            object
ProductName          object
EngineVersion        object
AppVersion           object
SignatureVersion     object
                     ...   
IsGamer             float64
RegionIdentifier    float64
DateAS               object
DateOS               object
target                int64
Length: 76, dtype: object

In [312]:
for col in df.columns:
    print(f"{col}:  {df[col].isna().sum()}")

MachineID:  0
ProductName:  0
EngineVersion:  0
AppVersion:  0
SignatureVersion:  0
IsBetaUser:  0
RealTimeProtectionState:  0
IsPassiveModeEnabled:  0
AntivirusConfigID:  0
NumAntivirusProductsInstalled:  0
NumAntivirusProductsEnabled:  0
HasTpm:  0
CountryID:  0
CityID:  0
GeoRegionID:  0
LocaleEnglishNameID:  0
PlatformType:  0
Processor:  0
OSVersion:  0
OSBuildNumber:  0
OSProductSuite:  0
OsPlatformSubRelease:  0
OSBuildLab:  0
SKUEditionName:  0
IsSystemProtected:  0
AutoSampleSubmissionEnabled:  0
SMode:  0
IEVersionID:  0
FirewallEnabled:  0
EnableLUA:  0
MDC2FormFactor:  0
DeviceFamily:  0
OEMNameID:  0
OEMModelID:  0
ProcessorCoreCount:  0
ProcessorManufacturerID:  0
ProcessorModelID:  0
PrimaryDiskCapacityMB:  0
PrimaryDiskType:  0
SystemVolumeCapacityMB:  0
HasOpticalDiskDrive:  0
TotalPhysicalRAMMB:  0
ChassisType:  0
PrimaryDisplayDiagonalInches:  0
PrimaryDisplayResolutionHorizontal:  0
PrimaryDisplayResolutionVertical:  0
PowerPlatformRole:  0
InternalBatteryNumberOfCh

In [313]:
df=df.drop('MachineID',axis=1)
print("MachineID dropped")

MachineID dropped


In [314]:
df['ProductName'].nunique()

2

In [315]:
df['ProductName'].value_counts()

ProductName
win8defender    99771
mse               229
Name: count, dtype: int64

In [316]:
df.groupby("ProductName")["target"].value_counts(normalize=True) * 100


ProductName   target
mse           0         54.585153
              1         45.414847
win8defender  1         50.536729
              0         49.463271
Name: proportion, dtype: float64

In [317]:
df["target"].value_counts(normalize=True) * 100


target
1    50.525
0    49.475
Name: proportion, dtype: float64

In [318]:
df = df.drop('ProductName',axis=1)
print("ProductName dropped")

ProductName dropped


In [319]:
count = df['EngineVersion'].str.startswith('1.1').sum()
print(count)

100000


In [320]:
df[['EngineVersion_BuildNo','EngineVersion_Revision']] = df['EngineVersion'].str.split('.',expand=True).iloc[:,2:].astype('int64')
df = df.drop('EngineVersion',axis=1)
print("EngineVersion dropped")

EngineVersion dropped


In [321]:
df['AppVersion'].str.startswith('4').sum()

100000

In [322]:
df[['AppVersion_MinorUpdate','AppVersion_BuildNo','AppVersion_Revision']] = df['AppVersion'].str.split('.',expand=True).iloc[:,1:].astype('int64')
df = df.drop('AppVersion',axis=1)
print("AppVersion dropped")

AppVersion dropped


In [323]:
df['SignatureVersion'].str.endswith('0').sum()

100000

In [324]:
df['SignatureVersion'].str.startswith('1.2').sum()

100000

In [325]:
df[['SignatureVersion_MinorUpdate','SignatureVersion_BuildNo']] = df['SignatureVersion'].str.split('.',expand=True).iloc[:,1:3].astype('int64')
df['SignatureVersion_MinorUpdate'] = df['SignatureVersion_MinorUpdate'] % 200
df = df.drop('SignatureVersion',axis=1)
print("SignatureVersion dropped")

SignatureVersion dropped


In [326]:
df['CityID'].value_counts()

CityID
130775.0    1769
82373.0      966
16668.0      966
10222.0      879
66202.0      772
            ... 
16718.0        1
87973.0        1
58209.0        1
102163.0       1
155850.0       1
Name: count, Length: 16047, dtype: int64

In [327]:
df['PlatformType'] = df['PlatformType'].str.extract('(\d+)$').astype('int64')


In [328]:
print(df['PlatformType'].value_counts())

PlatformType
10      98344
8        1401
7         226
2016       29
Name: count, dtype: int64


In [329]:
groups = [df[df['OSVersion']==cat]['PlatformType'] for cat in df['OSVersion'].unique()]
stats,p_value = f_oneway(*groups)
print(f"ANOVA p-value: {p_value}")

ANOVA p-value: 0.109006706708461


In [330]:
df['Processor'] = df['Processor'].str.extract('(\d+)$').astype('int64')

In [331]:
df['OSVersion'].str.endswith('0.0').sum()

99769

In [332]:
groups = [df[df['OSArchitecture'] == cat]['Processor'] for cat in df['OSArchitecture'].unique()]
stat, p_value = f_oneway(*groups)

print(f"ANOVA p-value: {p_value}")


ANOVA p-value: 0.0


In [333]:
df = df.drop('OSArchitecture',axis=1)
print("OSArchitecture dropped")

OSArchitecture dropped


In [335]:
df[['OSVersion_MinorUpdate','OSVersion_BuildNo']] = df['OSVersion'].str.split('.',expand=True).iloc[:,0:2].astype('int64')
df = df.drop('OSVersion',axis=1)

KeyError: 'OSVersion'

In [336]:
(df['OSBuildLab'].str.split('.',expand=True)[2].str[-3:]=='fre').sum()

100000

In [337]:
df['OSBuildLab_BuildNo']=df['OSBuildLab'].str.split('.',expand=True).iloc[:,1].astype('int64')

In [338]:
osbuildlab_datetime = df['OSBuildLab'].str.split('.').str[-1]
split_df = osbuildlab_datetime.str.split('-', expand=True)[[0, 1]]
osbuildlab_date = split_df[0]
osbuildlab_hhmm = split_df[1]
YY = osbuildlab_date.str[:2].astype(int)
MM = osbuildlab_date.str[2:4].astype(int)
DD = osbuildlab_date.str[4:6].astype(int)
HH = osbuildlab_hhmm.str[:2].astype(int)
mm = osbuildlab_hhmm.str[2:4].astype(int)

df['OSBuildLab_Time'] = (YY * 360 * 24 * 60 + MM * 30 * 24 * 60 + DD * 24 * 60 + HH * 60 + mm)
df['OSBuildLab_Time'] -= df['OSBuildLab_Time'].min()

df = df.drop('OSBuildLab',axis=1)

In [339]:
df['DeviceFamily'].value_counts()

DeviceFamily
Windows.Desktop    99971
Windows.Server        29
Name: count, dtype: int64

In [340]:
df.groupby('DeviceFamily')['target'].value_counts(normalize=True)*100

DeviceFamily     target
Windows.Desktop  1         50.531654
                 0         49.468346
Windows.Server   0         72.413793
                 1         27.586207
Name: proportion, dtype: float64

In [341]:
df['PrimaryDiskCapacityMB'].nunique()

398

In [342]:
def mutual_info_between_categorical(df, col1, col2):
    le1, le2 = LabelEncoder(), LabelEncoder()
    x_encoded = le1.fit_transform(df[col1])
    y_encoded = le2.fit_transform(df[col2])
    mi = mutual_info_classif(x_encoded.reshape(-1, 1), y_encoded, discrete_features=True)
    return mi[0]

mutual_info = mutual_info_between_categorical(df, 'OSEdition', 'OSSkuFriendlyName')
print(mutual_info)


1.2326021823638782


In [343]:
df = df.drop('OSSkuFriendlyName',axis=1)
print("OSSkuFriendlyName dropped")

OSSkuFriendlyName dropped


In [344]:
splitted_df = df['DateAS'].str.split('-',expand=True)
DD = splitted_df[0].astype('int64')
MM = splitted_df[1].astype('int64')
YY_time = splitted_df[2].str.split(' ',expand=True)
YY = YY_time[0].astype('int64')%2000
HH = YY_time[1].str[:2].astype('int64')
mm = YY_time[1].str[3:5].astype('int64')

df['DateAS']=(YY*360*24*60+MM*30*24*60+DD*24*60+HH*60+mm)
df['DateAS']-=df['DateAS'].min()

In [345]:
df['DateOS'].isna().sum()

0

In [346]:
print(type(df['DateOS'][0]))

<class 'str'>


In [347]:
splitted_df = df['DateOS'].astype(str).str.split('-',expand=True)
DD = pd.to_numeric(splitted_df[0], errors='coerce')
MM = pd.to_numeric(splitted_df[1], errors='coerce')
YY = pd.to_numeric(splitted_df[2], errors='coerce') % 2000

new_dateOS =(YY*360*24*60+MM*30*24*60+DD*24*60)
df['DateOS'] = new_dateOS - new_dateOS.min(skipna=True)

In [348]:
categ_cols = df.select_dtypes(include = ['object','category']).columns
print(categ_cols)

Index(['OsPlatformSubRelease', 'SKUEditionName', 'MDC2FormFactor',
       'DeviceFamily', 'PrimaryDiskType', 'ChassisType', 'PowerPlatformRole',
       'NumericOSVersion', 'OSBranch', 'OSEdition', 'OSInstallType',
       'AutoUpdateOptionsName', 'OSGenuineState', 'LicenseActivationChannel',
       'FlightRing'],
      dtype='object')


In [349]:
df = df.drop('NumericOSVersion',axis=1)

In [350]:
def mutual_info_between_categorical(df, col1, col2):
    le1, le2 = LabelEncoder(), LabelEncoder()
    x_encoded = le1.fit_transform(df[col1])
    y_encoded = le2.fit_transform(df[col2])
    mi = mutual_info_classif(x_encoded.reshape(-1, 1), y_encoded, discrete_features=True)
    return mi[0]

mutual_info = mutual_info_between_categorical(df, 'OsPlatformSubRelease', 'OSBranch')
print(mutual_info)

1.2820445938296559


In [351]:
df = df.drop('OsPlatformSubRelease',axis=1)
print("OsPlatformSubRelease dropped")

OsPlatformSubRelease dropped


In [352]:
categ_cols = df.select_dtypes(include = ['object']).columns
print(categ_cols)

Index(['SKUEditionName', 'MDC2FormFactor', 'DeviceFamily', 'PrimaryDiskType',
       'ChassisType', 'PowerPlatformRole', 'OSBranch', 'OSEdition',
       'OSInstallType', 'AutoUpdateOptionsName', 'OSGenuineState',
       'LicenseActivationChannel', 'FlightRing'],
      dtype='object')


In [353]:
for col in categ_cols:
    print(f"{col}:  {df[col].nunique()}")

SKUEditionName:  8
MDC2FormFactor:  11
DeviceFamily:  2
PrimaryDiskType:  4
ChassisType:  28
PowerPlatformRole:  9
OSBranch:  13
OSEdition:  20
OSInstallType:  9
AutoUpdateOptionsName:  6
OSGenuineState:  4
LicenseActivationChannel:  6
FlightRing:  7


In [354]:
for col in df.columns:
    print(f"{col}:  {type(df[col][0])}")

IsBetaUser:  <class 'numpy.int64'>
RealTimeProtectionState:  <class 'numpy.float64'>
IsPassiveModeEnabled:  <class 'numpy.int64'>
AntivirusConfigID:  <class 'numpy.float64'>
NumAntivirusProductsInstalled:  <class 'numpy.float64'>
NumAntivirusProductsEnabled:  <class 'numpy.float64'>
HasTpm:  <class 'numpy.int64'>
CountryID:  <class 'numpy.int64'>
CityID:  <class 'numpy.float64'>
GeoRegionID:  <class 'numpy.float64'>
LocaleEnglishNameID:  <class 'numpy.int64'>
PlatformType:  <class 'numpy.int64'>
Processor:  <class 'numpy.int64'>
OSBuildNumber:  <class 'numpy.int64'>
OSProductSuite:  <class 'numpy.int64'>
SKUEditionName:  <class 'str'>
IsSystemProtected:  <class 'numpy.float64'>
AutoSampleSubmissionEnabled:  <class 'numpy.int64'>
SMode:  <class 'numpy.float64'>
IEVersionID:  <class 'numpy.float64'>
FirewallEnabled:  <class 'numpy.float64'>
EnableLUA:  <class 'numpy.float64'>
MDC2FormFactor:  <class 'str'>
DeviceFamily:  <class 'str'>
OEMNameID:  <class 'numpy.float64'>
OEMModelID:  <cla

In [355]:
df.shape

(100000, 76)

## **Test Data Loading**
The test data was loaded and all the steps of data preparation applied on the training data were repeated on it.

In [380]:
X_test = pd.read_csv('/kaggle/input/System-Threat-Forecaster/test.csv')

imputer = SimpleImputer(strategy='most_frequent')  # Mode imputation
test_imputed = pd.DataFrame(imputer.fit_transform(X_test), columns=X_test.columns)

X_test = test_imputed.astype(X_test.dtypes.to_dict())
X_test=X_test.drop('MachineID',axis=1)
X_test = X_test.drop('ProductName',axis=1)
X_test[['EngineVersion_BuildNo','EngineVersion_Revision']] = X_test['EngineVersion'].str.split('.',expand=True).iloc[:,2:].astype('int64')
X_test = X_test.drop('EngineVersion',axis=1)
X_test[['AppVersion_MinorUpdate','AppVersion_BuildNo','AppVersion_Revision']] = X_test['AppVersion'].str.split('.',expand=True).iloc[:,1:].astype('int64')
X_test = X_test.drop('AppVersion',axis=1)
X_test[['SignatureVersion_MinorUpdate','SignatureVersion_BuildNo']] = X_test['SignatureVersion'].str.split('.',expand=True).iloc[:,1:3].astype('int64')
X_test['SignatureVersion_MinorUpdate'] = X_test['SignatureVersion_MinorUpdate'] % 200
X_test = X_test.drop('SignatureVersion',axis=1)
X_test['PlatformType'] = X_test['PlatformType'].str.extract('(\d+)$').astype('int64')
X_test['Processor'] = X_test['Processor'].str.extract('(\d+)$').astype('int64')
X_test = X_test.drop('OSArchitecture',axis=1)
X_test[['OSVersion_MinorUpdate','OSVersion_BuildNo']] = X_test['OSVersion'].str.split('.',expand=True).iloc[:,0:2].astype('int64')
X_test = X_test.drop('OSVersion',axis=1)
X_test['OSBuildLab_BuildNo']=X_test['OSBuildLab'].str.split('.',expand=True).iloc[:,1].astype('int64')
osbuildlab_datetime = X_test['OSBuildLab'].str.split('.').str[-1]
split_df = osbuildlab_datetime.str.split('-', expand=True)[[0, 1]]
osbuildlab_date = split_df[0]
osbuildlab_hhmm = split_df[1]
YY = osbuildlab_date.str[:2].astype(int)
MM = osbuildlab_date.str[2:4].astype(int)
DD = osbuildlab_date.str[4:6].astype(int)
HH = osbuildlab_hhmm.str[:2].astype(int)
mm = osbuildlab_hhmm.str[2:4].astype(int)

X_test['OSBuildLab_Time'] = (YY * 360 * 24 * 60 + MM * 30 * 24 * 60 + DD * 24 * 60 + HH * 60 + mm)
X_test['OSBuildLab_Time'] -= X_test['OSBuildLab_Time'].min()

X_test = X_test.drop('OSBuildLab',axis=1)
X_test = X_test.drop('OSSkuFriendlyName',axis=1)
splitted_df = X_test['DateAS'].str.split('-',expand=True)
DD = splitted_df[0].astype('int64')
MM = splitted_df[1].astype('int64')
YY_time = splitted_df[2].str.split(' ',expand=True)
YY = YY_time[0].astype('int64')%2000
HH = YY_time[1].str[:2].astype('int64')
mm = YY_time[1].str[3:5].astype('int64')

X_test['DateAS']=(YY*360*24*60+MM*30*24*60+DD*24*60+HH*60+mm)
X_test['DateAS']-=X_test['DateAS'].min()
splitted_df = X_test['DateOS'].astype(str).str.split('-',expand=True)
DD = pd.to_numeric(splitted_df[0], errors='coerce')
MM = pd.to_numeric(splitted_df[1], errors='coerce')
YY = pd.to_numeric(splitted_df[2], errors='coerce') % 2000

new_dateOS =(YY*360*24*60+MM*30*24*60+DD*24*60)
X_test['DateOS'] = new_dateOS - new_dateOS.min(skipna=True)
X_test = X_test.drop('NumericOSVersion',axis=1)
X_test = X_test.drop('OsPlatformSubRelease',axis=1)


## **Logistic Regression**
### **Data Preprocessing** 
The numeric features were scaled using z-score normalization because Logostic Regression assumes normal distribution of data. The categoric features were encoded using OrdinalEncoder.
### **Training**
GridSearchCV was performed to obtained the optimum values for Regularization Tolerance(C) and number of iterations(max_iter). The best model thus obtained was used to predict the test labels.
### **Highest Accuracy Obtained**
Highest Accuracy Obtained by this model was 0.6128

In [381]:
# Identify numeric and categorical columns
'''num_cols = df.select_dtypes(include=['number']).columns.tolist()
cat_cols = df.select_dtypes(include=['object']).columns.tolist()

categories = [df[col].unique().tolist() for col in cat_cols]
# Drop target column from feature lists
target = "target"
num_cols.remove(target)

# Preprocessing: Scaling numeric data, Encoding categorical data
num_transformer = StandardScaler()
cat_transformer = OrdinalEncoder(categories=categories)

preprocessor = ColumnTransformer([
    ('num', num_transformer, num_cols),
    ('cat', cat_transformer, cat_cols)
])

# Apply transformations
X = preprocessor.fit_transform(df.drop(columns=[target]))
y = df[target].values'''

In [ ]:
'''X = df.drop('target',axis=1)
y = df['target']

#X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train = X
y_train = y

cat_cols = df.select_dtypes(include = ['object','category']).columns.tolist()
categories = [X[col].unique().tolist() for col in cat_cols]

encoder = OrdinalEncoder(categories=categories)

X_train[cat_cols] = encoder.fit_transform(X_train[cat_cols])
X_test[cat_cols] = encoder.transform(X_test[cat_cols])

scaler = StandardScaler()

# Fit on X_train and transform both X_train and X_test
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)'''

In [360]:
#X.shape

(100000, 189)

In [377]:
'''feature_names=df.columns
X_test_transformed = pd.DataFrame(X_test, columns=feature_names)
for col in cat_cols:
    if df[col].nunique()!=X_test_transformed[col].nunique():
        print(col)'''

SKUEditionName
MDC2FormFactor
ChassisType
PowerPlatformRole
OSBranch
OSEdition
LicenseActivationChannel


In [383]:
# Define model
#lr = LogisticRegression(solver='liblinear')

# Hyperparameter grid
param_grid = {
    'C': 10,#[0.01, 0.1, 1, 10, 100],  
    'max_iter': 100#[100, 200, 500]
}

# Grid Search with Cross Validation
#grid_search = GridSearchCV(lr, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
#grid_search.fit(X_train, y_train)

# Get best parameters and save to file
#best_params = grid_search.best_params_
#with open("best_params.txt", "w") as f:
#    f.write(str(best_params))

# Train best model
#best_model = grid_search.best_estimator_

In [366]:
#type(X_test)

pandas.core.frame.DataFrame

In [384]:
# Identify numeric and categorical columns
'''num_cols = X_test.select_dtypes(include=['number']).columns.tolist()
cat_cols = X_test.select_dtypes(include=['object']).columns.tolist()

# Preprocessing: Scaling numeric data, Encoding categorical data
num_transformer = StandardScaler()
cat_transformer = OneHotEncoder(handle_unknown='ignore')

preprocessor = ColumnTransformer([
    ('num', num_transformer, num_cols),
    ('cat', cat_transformer, cat_cols)
])'''

# Apply transformations
#X_test = preprocessor.transform(X_test[cat_cols])

KeyError: "None of [Index(['IsBetaUser', 'RealTimeProtectionState', 'IsPassiveModeEnabled',\n       'AntivirusConfigID', 'NumAntivirusProductsInstalled',\n       'NumAntivirusProductsEnabled', 'HasTpm', 'CountryID', 'CityID',\n       'GeoRegionID', 'LocaleEnglishNameID', 'PlatformType', 'Processor',\n       'OSBuildNumber', 'OSProductSuite', 'IsSystemProtected',\n       'AutoSampleSubmissionEnabled', 'SMode', 'IEVersionID',\n       'FirewallEnabled', 'EnableLUA', 'OEMNameID', 'OEMModelID',\n       'ProcessorCoreCount', 'ProcessorManufacturerID', 'ProcessorModelID',\n       'PrimaryDiskCapacityMB', 'SystemVolumeCapacityMB',\n       'HasOpticalDiskDrive', 'TotalPhysicalRAMMB',\n       'PrimaryDisplayDiagonalInches', 'PrimaryDisplayResolutionHorizontal',\n       'PrimaryDisplayResolutionVertical', 'InternalBatteryNumberOfCharges',\n       'OSBuildNumberOnly', 'OSBuildRevisionOnly', 'OSInstallLanguageID',\n       'OSUILocaleID', 'IsPortableOS', 'IsFlightsDisabled',\n       'FirmwareManufac

In [371]:
# Train Logistic Regression with optimal parameters using best_params
'''best_model = LogisticRegression(**param_grid, solver='liblinear')
best_model.fit(X, y)

# Predict class labels (0 or 1) for X_test
y_pred = best_model.predict(X_test)

result=pd.DataFrame({
    'id':range(0,X_test.shape[0]),
    'target':y_pred
})
result.to_csv('submission.csv', index=False)'''

ValueError: X has 171 features, but LogisticRegression is expecting 189 features as input.

## **XGBoost**
### **Data Preprocessing**
Ordinal Encoding was used for the categorical columns in both the training and test data. To ensure uniform encoding across the training and test data, a list of the unique values of each categoric column was maintained and the encoding across the training and test data was done based on it. Tree-based methods are eligible for ordinal encoding because the absolute value of features does not affect their classification.
### **Training**
At first, the data was split into training and test in 80:20 ratio. The training data was used to train the model and GridSearchCv was performed to obtain the optimum values for the hyperparameters of the model. The optimum hyperparameters thus obtained were used to train the model and the importance scores for each of the feature was obtained from the trained model. Next, the features were arranged in descending order of importance scores and the top k features from both training and test sets were utilized for training and accuracy measurement while iterating over k. This way, it was observed that taking the top 59 features gave the greatest test accuracy. This information was then utilized on the test data provided and the label prediction was made.
### **Highest Accuracy Obtained**
0.6324

In [385]:
X = df.drop('target',axis=1)
y = df['target']

#X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train = X
y_train = y

cat_cols = df.select_dtypes(include = ['object','category']).columns.tolist()

In [ ]:
for col in df.columns:
    print(f"{col}:  {type(df[col][0])}")

In [386]:
categories = [X[col].unique().tolist() for col in cat_cols]

encoder = OrdinalEncoder(categories=categories)

X_train[cat_cols] = encoder.fit_transform(X_train[cat_cols])
X_test[cat_cols] = encoder.transform(X_test[cat_cols])

In [387]:
param_grid = {
    'n_estimators': 300,
    'max_depth': 6,
    'learning_rate': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'gamma': 0.5
}
# Initialize XGBoost model
xgb_model = XGBClassifier(**param_grid,use_label_encoder=False, eval_metric='logloss')
xgb_model.fit(X_train, y_train)



In [ ]:
# Get feature importance scores
feature_importances = pd.DataFrame({'Feature': X_train.columns, 'Importance': xgb_model.feature_importances_})
feature_importances = feature_importances.sort_values(by='Importance', ascending=False)

In [ ]:
top_k_features = feature_importances.iloc[:59]['Feature'].values
X_train_k = X_train[top_k_features]
X_test_k = X_test[top_k_features]

xgb_model.fit(X_train_k, y_train)
y_pred = xgb_model.predict(X_test_k)

In [ ]:
result=pd.DataFrame({
    'id':range(0,X_test.shape[0]),
    'target':y_pred
})
result.to_csv('submission.csv', index=False)

## **Random Forest**
### **Data Preparation**
Ordinal Encoding was used for the categorical columns in both the training and test data. To ensure uniform encoding across the training and test data, a list of the unique values of each categoric column was maintained and the encoding across the training and test data was done based on it. Tree-based methods are eligible for ordinal encoding because the absolute value of features does not affect their classification.
### **Training**
At first, the data was split into training and test in 80:20 ratio. The training data was used to train the model and GridSearchCv was performed to obtain the optimum values for the hyperparameters of the model. The optimum hyperparameters thus obtained were used to train the model and the importance scores for each of the feature was obtained from the trained model. Next, the features were arranged in descending order of importance scores and the top k features from both training and test sets were utilized for training and accuracy measurement while iterating over k. This way, it was observed that taking the top 39 features gave the greatest test accuracy. This information was then utilized on the test data provided and the label prediction was made.
### **Highest Accuracy Obtained**
0.6285

In [ ]:
'''rf_model = RandomForestClassifier(random_state=42)

param_grid = {
    'n_estimators': [100, 200, 300],  # Balances performance and speed
    'max_depth': [15, 20, 30],  # Prevents overfitting
    'min_samples_split': [10, 20, 30],  # Ensures meaningful splits
    'min_samples_leaf': [5, 10, 15],  # Prevents tiny leaves
    'max_features': ['sqrt', 'log2']  # Controls feature selection per tree
}

grid_search = GridSearchCV(rf_model, param_grid, scoring='accuracy', cv=5, n_jobs=-1)
grid_search.fit(X_train, y_train)

best_rf = grid_search.best_estimator_

y_pred = best_rf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Best Parameters: {grid_search.best_params_}")
print(f"Test Accuracy: {accuracy:.4f}")'''

In [ ]:
'''print("X_train:  ")
for col in X_train.columns:
    print(f"{col}:  {X_train[col].isna().sum()}")

print("\n\n\nX_test:  ")
for col in X_test.columns:
    print(f"{col}:  {X_test[col].isna().sum()}")'''

In [ ]:
'''X = df.drop('target',axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

categ_cols = df.select_dtypes(include = ['object','category']).columns.tolist()
catboost_model = CatBoostClassifier(verbose=0)

param_grid = {
    'iterations': [500, 1000],
    'depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'l2_leaf_reg': [1, 3, 5],
    'border_count': [32, 64, 128]
}

grid_search = GridSearchCV(catboost_model, param_grid, scoring='accuracy', cv=5, n_jobs=-1)
grid_search.fit(X_train, y_train)

best_params = grid_search.best_params_
print("Best Parameters:", best_params)

best_catboost = CatBoostClassifier(**best_params, verbose=0)
best_catboost.fit(X_train, y_train, cat_features=categ_cols)

accuracy = best_catboost.score(X_test, y_test)
print("Test Accuracy:", accuracy)'''


In [ ]:
'''rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=30,
    min_samples_split=30,
    min_samples_leaf=5,
    max_features='sqrt',
    random_state=42 
)

rf_model.fit(X_train, y_train)'''

#y_pred = rf_model.predict(X_test)#highest accuracy for top 39 features

'''result=pd.DataFrame({
    'id':range(0,X_test.shape[0]),
    'target':y_pred
})
result.to_csv('submission.csv', index=False)'''

In [ ]:
# Train the best XGBoost model using the best parameters
'''best_xgb = xgb.XGBClassifier(**best_params, use_label_encoder=False, eval_metric='logloss')
best_xgb.fit(X_train, y_train)'''

In [ ]:
# Get feature importance scores
'''feature_importances = pd.DataFrame({'Feature': X_train.columns, 'Importance': rf_model.feature_importances_})
feature_importances = feature_importances.sort_values(by='Importance', ascending=False)

print(feature_importances)'''

In [ ]:
# Define parameter grid
'''param_grid = {
    'n_estimators': [100, 300],
    'max_depth': [3, 6, 10],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'gamma': [0, 1, 5]
}

# Initialize XGBoost model
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')

# Perform Grid Search with 5-fold cross-validation
grid_search = GridSearchCV(xgb_model, param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)

# Save best parameters to a file
best_params = grid_search.best_params_
with open("best_xgb_params.txt", "w") as f:
    f.write(str(best_params))'''

In [ ]:
'''best_accuracy = 0
best_k = 0
accuracies = []

# Try different numbers of top features
for k in range(1, len(X_train.columns) + 1):
    top_k_features = feature_importances.iloc[:k]['Feature'].values
    X_train_k = X_train[top_k_features]
    X_test_k = X_test[top_k_features]

    best_xgb.fit(X_train_k, y_train)
    y_pred = best_xgb.predict(X_test_k)
    
    acc = accuracy_score(y_test, y_pred)
    accuracies.append((k, acc))

    print(f"Accuracy: {acc} with {k} features")

    if acc > best_accuracy:
        best_accuracy = acc
        best_k = k

print(f"Best accuracy: {best_accuracy} with {best_k} features")

acc_result = pd.DataFrame(accuracies, columns=['k','Accuracy'])
acc_result.to_csv('Accuracies.csv',index = False)'''

In [ ]:
'''best_accuracy = 0
best_k = 0
accuracies = []

# Try different numbers of top features
for k in range(1, len(X_train.columns) + 1):
    top_k_features = feature_importances.iloc[:k]['Feature'].values
    X_train_k = X_train[top_k_features]
    X_test_k = X_test[top_k_features]

    rf_model.fit(X_train_k, y_train)
    y_pred = rf_model.predict(X_test_k)
    
    acc = accuracy_score(y_test, y_pred)
    accuracies.append((k, acc))

    print(f"Accuracy: {acc} with {k} features")

    if acc > best_accuracy:
        best_accuracy = acc
        best_k = k

print(f"Best accuracy: {best_accuracy} with {best_k} features")

acc_result = pd.DataFrame(accuracies, columns=['k','Accuracy'])
acc_result.to_csv('Accuracies.csv',index = False)'''

In [ ]:
'''df_test = pd.read_csv('/kaggle/input/System-Threat-Forecaster/test.csv')

for col in df_test.columns:
    print(f"{col}:  {df_test[col].isna().sum()}")'''

In [ ]:
#df=df.drop('MachineID',axis=1)


In [ ]:
#y_pred = rf_model.predict(X_test)